# Preprocess MUG

In [ ]:
import tensorflow as tf

import os
from pathlib import Path
import time
import datetime
import numpy as np
import cv2
import hashlib

from matplotlib import pyplot as plt
from IPython import display
from tqdm.notebook import tqdm

In [ ]:
# TODO: video path to hash

def repeat_last_frame_to_fill(video, n):
    if len(video) < n:
        video = np.concatenate([video, np.repeat(video[-1:], n-len(video), axis=0)], axis=0) # TODO: check axis
    return video

def get_clip_path(folder, video_number, clip_number):
    return  folder / f"{video_number}" / f"{clip_number}.npy"

def load_clip(cap, n=None):
    if n is None:
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    return np.array([cap.read()[1] for i in range(n)])

# TODO: check if file exist from hash
# TODO: this function suposes if it is the same path is the exact same video
def load_video(path, clip_size=20):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    clips = []
    video = load_clip(cap, n)
    if n > 20: 
        # split video into 20 frame chunks
        for i in range(0, n, clip_size):
            current_clip = video[i:i+clip_size]
            if len(current_clip) < clip_size:
                current_clip = repeat_last_frame_to_fill(current_clip, clip_size)
            clips.append(video[i:i+clip_size])
    else:
        clips.append(repeat_last_frame_to_fill(video, clip_size))
    return clips

# resize video lanczos4
def resize_video(video, size=64):
    return np.array([cv2.resize(frame, (size,size), interpolation=cv2.INTER_LANCZOS4) for frame in video])

# BRG to RGB
def bgr_to_rgb(video):
    return np.array([cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in video])

def save_video(video, path, video_number):
    for i, clip in enumerate(video):
        clip_path = get_clip_path(path, video_number, i)
        clip_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(str(clip_path), clip)
        
# load videos from npy
def load_videos(videos_paths, n=None):
    videos_paths = list(videos_paths)
    if n is None:
        n = len(videos_paths)
    else:
        videos_paths = videos_paths[:n]
    videos = []
    for path in tqdm(videos_paths, total=n):
        videos.append(np.load(path))
    return np.array(videos, dtype=np.float32)

def process_videos(videos_paths, save_path):
    videos_paths = list(videos_paths)
    n = len(videos_paths)
    for i, video_path in tqdm(enumerate(videos_paths), total=n):
        video = load_video(video_path)
        clips = np.array(list(map(resize_video, video)))
        clips = np.array(list(map(bgr_to_rgb, clips)))
        save_video(clips, save_path, i)


In [ ]:
mug_path = Path('../data/mug/')
save_path = Path('../data/').resolve() / 'mug_clips'
process_videos(list(mug_path.glob('**/*.avi')), save_path)

In [ ]:
sequences = load_videos(save_path.glob('**/*.npy'), n=100)
print(sequences.shape)